# ⚽ Highlights de Martin — Motor en Google Colab

Ejecuta las celdas con GPU. Son idempotentes: puedes volver a ejecutarlas.

> Si Colab muestra **Conectando** durante varios minutos, usa *Entorno de ejecución → Desconectar y eliminar entorno de ejecución*, vuelve a conectar y ejecuta otra vez.


In [ ]:
# 1) Comprobar la GPU
!nvidia-smi

In [ ]:
# 2) Traer o actualizar el código oficial
%cd /content
![ -d martin-highlights/.git ] && git -C martin-highlights pull --ff-only || git clone https://github.com/vjcano-gif/martin-highlights.git
%cd /content/martin-highlights


In [ ]:
# 3) Instalar dependencias (ffmpeg ya viene en Colab)
!pip install -q -r requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version


## 4) Lanzar la app

⚠️ **El enlace de Cloudflare Quick Tunnel es público.** No compartas la URL ni subas cookies que no estés dispuesto a usar durante esta sesión. El enlace deja de funcionar al desconectar Colab.


In [ ]:
# Verifica que Streamlit todavía no esté ocupando el puerto (es normal que no muestre nada)
!fuser 8501/tcp 2>/dev/null || true


In [ ]:
# Reinicia la app y crea un túnel nuevo y estable
!pkill -f 'streamlit run app.py' || true
!pkill -f 'cloudflared tunnel' || true
!nohup streamlit run app.py --server.headless true --server.address 127.0.0.1 --server.port 8501 >/content/logs.txt 2>&1 &

import pathlib, re, subprocess, time, urllib.request
from IPython.display import Markdown, display

# Espera a que Streamlit responda antes de publicar el túnel.
for attempt in range(30):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Streamlit no inició. Ejecuta: !cat /content/logs.txt')

tunnel_log = open('/content/tunnel.log', 'w')
subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8501', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    log = pathlib.Path('/content/tunnel.log').read_text(errors='replace')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
    if match:
        tunnel_url = match.group(0)
        break
if not tunnel_url:
    raise RuntimeError('No se creó el túnel. Ejecuta: !cat /content/tunnel.log')
display(Markdown(f'## ✅ [Abrir Highlights de Martin]({tunnel_url})'))
print('Usa solo este enlace trycloudflare.com. Cierra cualquier pestaña antigua de loca.lt.')


### ¿Problemas?
- **No reutilices una dirección `loca.lt` anterior:** puede servir archivos JavaScript desactualizados y causar `Failed to fetch dynamically imported module`. Usa solamente la nueva URL `trycloudflare.com`.
- Abre el enlace en una pestaña nueva. Si el navegador conserva una versión anterior, recarga con `Ctrl+Shift+R`.
- Si aparece **Conectando** antes de ejecutar una celda, reinicia el entorno desde el menú de Colab.
- Si YouTube bloquea la descarga, exporta un `cookies.txt` desde tu navegador y súbelo en *Opciones avanzadas*.
- Para ver errores de la app: `!cat /content/logs.txt`. Para el túnel: `!cat /content/tunnel.log`.
